In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments,T5ForConditionalGeneration

In [ ]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

print(train_data.shape)
print(val_data.shape)

(14732, 3)
(818, 3)


In [4]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [5]:
train_data.shape

(14732, 3)

In [6]:
val_data.shape

(818, 3)

In [7]:
#Random sampling

train_data=train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [8]:
train_data.shape

(4000, 3)

# Data pre-processing

In [9]:
import re

def clean_data(text):
    text = re.sub(r"\r\n"," ",text)  #lines
    text = re.sub(r"\s+"," ",text)  #spaces
    text = re.sub(r"<.*?>"," ",text)  #html tags
    text = text.strip().lower()
    return text

In [10]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [11]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

# Tokenization

In [12]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [13]:
# raw data => tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"]=targets["input_ids"] #token ids=>add to input as labels
    return inputs

In [14]:
train_dataset = train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()

# Working with our model

In [15]:
#NLP => generation task
model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [16]:
import torch

if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: ",device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [17]:
# tarining arguments

training_args = TrainingArguments(
    output_dir = "./results", #Training se related output aur checkpoints save karne ke liye folder
    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
)

In [18]:
trainer = Trainer(
    model = model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [19]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.624969,0.380839
2,0.397410,0.360152
3,0.374082,0.354482
4,0.362972,0.350377
5,0.355396,0.349707
6,0.351204,0.349091


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9110055898030599, metrics={'train_runtime': 1250.2562, 'train_samples_per_second': 19.196, 'train_steps_per_second': 2.4, 'total_flos': 3248203235328000.0, 'train_loss': 0.9110055898030599, 'epoch': 6.0})

In [20]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [22]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

# Test the core logic for summarization

In [25]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue) #clean

  # tokenizer
  inputs = tokenizer(
      dialogue,
      max_length=512,
      padding="max_length",
      truncation=True,
      return_tensors="pt"
  ).to(device)

  # generate the summary =>token ids
  model.to(device)
  targets = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length=150,
      num_beams=4,
      early_stopping=True
  )

  # token ids convert to summary => decoding
  summary = tokenizer.decode(
      targets[0],
      skip_special_tokens=True,
      # clean_up_tokenization_spaces=True
  )

  return summary

In [26]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, transforming the way people work and communicate.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and increase productivity across different sectors.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These developments are helping businesses solve complex problems more efficiently.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the limitations and biases present in their training data.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and responsible use of artificial intelligence.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, can produce accurate predictions without making their reasoning easy for humans to understand.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, transparency, and accountability.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be essential to ensure that AI benefits society while minimizing its risks.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ",summary)

Summary:  ai systems are becoming more capable due to advances in deep learning and access to large datasets. experts highlight the importance of responsible ai development, including data privacy, security, transparency and accountability.
